This notebook process our baselines (MolGlue-DB and PROTAC-DB 3.0).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def clean_compound_name(df, col='Compound_Name'):
    df = df.copy()
    df[col] = df[col].str.replace(r'^cmpd\s+', '', regex=True)
    df[col] = df[col].str.replace(r'^cmp', '', regex=True)
    return df


def standardize_dc50_h(df, col='DC50_h'):
    df = df.copy()
    df[col] = df[col].astype(str).str.replace(r'\s*h\s*$', '', regex=True)
    df[col] = df[col].replace({'nan': np.nan, '': np.nan})
    return df


def clear_orphan_dc50_units(df, dc50_col='DC50', units_col='DC50_units'):
    df = df.copy()
    mask = df[dc50_col].isna() & df[units_col].notna()
    n = int(mask.sum())
    df.loc[mask, units_col] = np.nan
    print(f"Cleared {n} {units_col} values where {dc50_col} was empty")
    return df


def filter_empty_dc50_dmax(df, cols=('DC50', 'DC50_units', 'DC50_h', 'Dmax', 'Dmax_h', 'Dmax_conc')):
    cols = list(cols)
    mask = df[cols].isna().all(axis=1)
    filtered = df[~mask].copy()
    print(f"Removed {mask.sum()} rows, {len(filtered)} rows remaining")
    return filtered


def add_connectivity_key(df, inchikey_col):
    df = df.copy()
    df['Connectivity_Key'] = df[inchikey_col].str[:14]
    return df


def normalize_doi(doi):
    doi = doi.astype(str).str.strip().str.lower()
    doi = doi.str.replace(r'^https?://(dx\.)?doi\.org/', '', regex=True)
    doi = doi.str.replace(r'^doi:\s*', '', regex=True)
    doi = doi.str.rstrip('/')
    doi = doi.replace({'nan': np.nan, 'none': np.nan, '': np.nan})
    return doi


def remove_excluded_dois(df, excluded_csv_path):
    excluded = pd.read_csv(excluded_csv_path)
    excluded_doi_norm = normalize_doi(excluded["DOI"])
    excluded_set = set(excluded_doi_norm.dropna())

    doi_norm = normalize_doi(df["DOI"])
    mask = doi_norm.isin(excluded_set)
    n_unique_matched = doi_norm[mask].nunique()

    before = len(df)
    filtered = df[~mask].reset_index(drop=True)
    after = len(filtered)

    print(f"Removed {before - after} rows ({n_unique_matched} unique excluded DOIs matched)")
    print(f"Shape: {before} -> {after}")
    return filtered

def split_multi_doi_rows(df, doi_col='DOI', sep=';'):
    df = df.copy()
    before = len(df)
    multi_mask = df[doi_col].astype(str).str.contains(sep, na=False)
    n_multi = int(multi_mask.sum())

    df[doi_col] = df[doi_col].astype(str).str.split(sep)
    df = df.explode(doi_col, ignore_index=True)
    df[doi_col] = df[doi_col].str.strip()

    print(f"Split {n_multi} multi-DOI rows; {before} -> {len(df)} rows after expansion")
    return df

def convert_excluded_dois_txt_to_csv(excluded_txt_path, excluded_csv_path):
    with open(excluded_txt_path) as f:
        lines = f.readlines()

    records = []
    current_reason = None
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if line.startswith("#"):
            if "NOT FOUND IN PMC DATABASE" in line:
                current_reason = "NOT_FOUND_IN_PMC_DATABASE"
            elif "IN PMC BUT NOT OPEN ACCESS" in line:
                current_reason = "IN_PMC_BUT_NOT_OPEN_ACCESS"
            continue
        if current_reason is None:
            continue
        for doi in line.split(";"):
            doi = doi.strip()
            if doi:
                records.append({"DOI": doi, "exclude_reason": current_reason})

    excluded_dois_df = pd.DataFrame(records)
    excluded_dois_df.to_csv(excluded_csv_path, index=False)
    print(f"Saved excluded DOIs to: {excluded_csv_path}")
    print(f"Shape: {excluded_dois_df.shape}")
    print(excluded_dois_df["exclude_reason"].value_counts())
    return excluded_dois_df

## PROCESS MOLECULAR GLUES BASELINE (MolGlue-DB)

In [2]:
glue_path = '/Users/yaochenr/project/tpd_curator/data/molecular_glues/baseline/MolGlueDB-baseline-all.csv'

In [3]:
glue_df = pd.read_csv(glue_path)
glue_df = clean_compound_name(glue_df)
glue_df

,Record_ID,Compound_ID,Compound_Name,SMILES,StdInChl,StdInChIKey,Degradation_Target,Recruiter,Recruiter_UniProtID,Cell_Line,Assay,DC50,DC50_units,DC50_h,Dmax,Dmax_h,Dmax_conc,DOI
0,810,747,FPFT-2216,COC1=CSC=C1C1=CN(C2CCC(=O)NC2=O)N=N1,InChI=1S/C12H12N4O3S/c1-19-10-6-20-5-7(10)8-4-...,SKUSCUALKIOIST-UHFFFAOYSA-N,CK1α,CRBN,Q96SW2,NaN,NaN,<0.01,μM,NaN,NaN,NaN,NaN,https://doi.org/10.1016/j.bmcl.2025.130193
1,810,747,FPFT-2216,COC1=CSC=C1C1=CN(C2CCC(=O)NC2=O)N=N1,InChI=1S/C12H12N4O3S/c1-19-10-6-20-5-7(10)8-4-...,SKUSCUALKIOIST-UHFFFAOYSA-N,IKZF1,CRBN,Q96SW2,NaN,NaN,<0.01,μM,NaN,NaN,NaN,NaN,https://doi.org/10.1016/j.bmcl.2025.130193
2,920,825,Pomalidomide,NC1=CC=CC2=C1C(=O)N(C1CCC(=O)NC1=O)C2=O,InChI=1S/C13H11N3O4/c14-7-3-1-2-6-10(7)13(20)1...,UVSMNLNDYGZFPF-UHFFFAOYSA-N,PLZF,CRBN,Q96SW2,NaN,NaN,<100,nM,NaN,NaN,NaN,NaN,https://doi.org/10.1038/s42003-021-02801-y
3,1215,1049,TMX-4100,O=C1CCC(N2C=C(C3=CSC=C3)N=N2)C(=O)N1,InChI=1S/C11H10N4O2S/c16-10-2-1-9(11(17)12-10)...,PGEUBQZQGXWTFO-UHFFFAOYSA-N,PDE6D,CRBN,Q96SW2,NaN,NaN,<200,nM,NaN,NaN,NaN,NaN,https://doi.org/10.1021/acs.jmedchem.1c01832
4,1463,1263,LC-03-041,O=C1CC[C@H](N2CC3=CC(C4=CN=C5CNCCN45)=CC=C3C2=...,InChI=1S/C19H19N5O3/c25-17-4-3-14(18(26)22-17)...,WDUXNRRGWRVGDQ-AWEZNQCLSA-N,NEK7,CRBN,Q96SW2,MOLT-4,HiBiT,> 1,μM,NaN,50,6 h,NaN,https://doi.org/10.1002/anie.202500169
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2569,82,77,HQ019,CC(=O)NC1=NC(CC(=O)NC2=NC=C(C)S2)=CS1,InChI=1S/C11H12N4O2S2/c1-6-4-12-10(19-6)15-9(1...,AVKUTTYOGRAGFM-UHFFFAOYSA-N,NaN,DDB1,Q16531,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://doi.org/10.7554/eLife.59994
2570,841,774,HQ021,NC(=O)C1=CC=CC=C1NC(=O)CC1=CSC(NC2=NC=CC=C2)=N1,InChI=1S/C17H15N5O2S/c18-16(24)12-5-1-2-6-13(1...,IOFFPPJYBLBCGR-UHFFFAOYSA-N,NaN,DDB1,Q16531,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://doi.org/10.7554/eLife.59994
2571,843,776,HQ022,NC(=O)CNC(=O)CC1=CSC(NC2=NC=CC=C2)=N1,InChI=1S/C12H13N5O2S/c13-9(18)6-15-11(19)5-8-7...,LNJMZTRTZHJTHB-UHFFFAOYSA-N,NaN,DDB1,Q16531,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://doi.org/10.7554/eLife.59994
2572,952,831,HQ013,NC1=NC(CC(=O)NC2=NC(C3=CC=CO3)=CS2)=CS1,InChI=1S/C12H10N4O2S2/c13-11-14-7(5-19-11)4-10...,CGQOACVDRKDGKA-UHFFFAOYSA-N,NaN,DDB1,Q16531,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://doi.org/10.7554/eLife.59994


### Remove rows with empty DC50 & Dmax

In [4]:
glue_df = clear_orphan_dc50_units(glue_df)
glue_df = filter_empty_dc50_dmax(glue_df)
glue_df.head()

Cleared 0 DC50_units values where DC50 was empty
Removed 2088 rows, 486 rows remaining


,Record_ID,Compound_ID,Compound_Name,SMILES,StdInChl,StdInChIKey,Degradation_Target,Recruiter,Recruiter_UniProtID,Cell_Line,Assay,DC50,DC50_units,DC50_h,Dmax,Dmax_h,Dmax_conc,DOI
0,810,747,FPFT-2216,COC1=CSC=C1C1=CN(C2CCC(=O)NC2=O)N=N1,InChI=1S/C12H12N4O3S/c1-19-10-6-20-5-7(10)8-4-...,SKUSCUALKIOIST-UHFFFAOYSA-N,CK1α,CRBN,Q96SW2,NaN,NaN,<0.01,μM,NaN,NaN,NaN,NaN,https://doi.org/10.1016/j.bmcl.2025.130193
1,810,747,FPFT-2216,COC1=CSC=C1C1=CN(C2CCC(=O)NC2=O)N=N1,InChI=1S/C12H12N4O3S/c1-19-10-6-20-5-7(10)8-4-...,SKUSCUALKIOIST-UHFFFAOYSA-N,IKZF1,CRBN,Q96SW2,NaN,NaN,<0.01,μM,NaN,NaN,NaN,NaN,https://doi.org/10.1016/j.bmcl.2025.130193
2,920,825,Pomalidomide,NC1=CC=CC2=C1C(=O)N(C1CCC(=O)NC1=O)C2=O,InChI=1S/C13H11N3O4/c14-7-3-1-2-6-10(7)13(20)1...,UVSMNLNDYGZFPF-UHFFFAOYSA-N,PLZF,CRBN,Q96SW2,NaN,NaN,<100,nM,NaN,NaN,NaN,NaN,https://doi.org/10.1038/s42003-021-02801-y
3,1215,1049,TMX-4100,O=C1CCC(N2C=C(C3=CSC=C3)N=N2)C(=O)N1,InChI=1S/C11H10N4O2S/c16-10-2-1-9(11(17)12-10)...,PGEUBQZQGXWTFO-UHFFFAOYSA-N,PDE6D,CRBN,Q96SW2,NaN,NaN,<200,nM,NaN,NaN,NaN,NaN,https://doi.org/10.1021/acs.jmedchem.1c01832
4,1463,1263,LC-03-041,O=C1CC[C@H](N2CC3=CC(C4=CN=C5CNCCN45)=CC=C3C2=...,InChI=1S/C19H19N5O3/c25-17-4-3-14(18(26)22-17)...,WDUXNRRGWRVGDQ-AWEZNQCLSA-N,NEK7,CRBN,Q96SW2,MOLT-4,HiBiT,> 1,μM,NaN,50,6 h,NaN,https://doi.org/10.1002/anie.202500169


### Add Connectivity_Key

In [5]:
glue_df = add_connectivity_key(glue_df, 'StdInChIKey')
glue_df[['Compound_Name', 'StdInChIKey', 'Connectivity_Key']]

,Compound_Name,StdInChIKey,Connectivity_Key
0,FPFT-2216,SKUSCUALKIOIST-UHFFFAOYSA-N,SKUSCUALKIOIST
1,FPFT-2216,SKUSCUALKIOIST-UHFFFAOYSA-N,SKUSCUALKIOIST
2,Pomalidomide,UVSMNLNDYGZFPF-UHFFFAOYSA-N,UVSMNLNDYGZFPF
3,TMX-4100,PGEUBQZQGXWTFO-UHFFFAOYSA-N,PGEUBQZQGXWTFO
4,LC-03-041,WDUXNRRGWRVGDQ-AWEZNQCLSA-N,WDUXNRRGWRVGDQ
...,...,...,...
1837,Lenalidomide,GOTYRUGSSMKFNF-UHFFFAOYSA-N,GOTYRUGSSMKFNF
1843,Br-Le,IIAHEYPYLVKWSS-UHFFFAOYSA-N,IIAHEYPYLVKWSS
1844,F3CO-Le,CFLQOYFUGDVTBH-UHFFFAOYSA-N,CFLQOYFUGDVTBH
1845,Br-Le,IIAHEYPYLVKWSS-UHFFFAOYSA-N,IIAHEYPYLVKWSS


## Save data

In [6]:
glue_df.to_csv('/Users/yaochenr/project/tpd_curator/data/molecular_glues/baseline/MolGlueDB-baseline-cleaned.csv', index=False)

## Remove DOIs not full-text avaliable in PMC

In [7]:
excluded_txt_path_glue = "/Users/yaochenr/project/tpd_curator/data_source/250825_molglue_papers/excluded_dois.txt"
excluded_csv_path_glue = "/Users/yaochenr/project/tpd_curator/data_source/250825_molglue_papers/excluded_dois.csv"

excluded_dois_df_glue = convert_excluded_dois_txt_to_csv(excluded_txt_path_glue, excluded_csv_path_glue)
excluded_dois_df_glue.head()

Saved excluded DOIs to: /Users/yaochenr/project/tpd_curator/data_source/250825_molglue_papers/excluded_dois.csv
Shape: (123, 2)
exclude_reason
NOT_FOUND_IN_PMC_DATABASE     85
IN_PMC_BUT_NOT_OPEN_ACCESS    38
Name: count, dtype: int64


,DOI,exclude_reason
0,10.1016/j.mencom.2022.11.013,NOT_FOUND_IN_PMC_DATABASE
1,10.1021/acs.jmedchem.9b01928,NOT_FOUND_IN_PMC_DATABASE
2,10.1016/j.cell.2024.10.015,NOT_FOUND_IN_PMC_DATABASE
3,10.1038/35104500,NOT_FOUND_IN_PMC_DATABASE
4,10.1038/nature05731,NOT_FOUND_IN_PMC_DATABASE


In [8]:
glue_df = remove_excluded_dois(glue_df, excluded_csv_path_glue)

Removed 350 rows (37 unique excluded DOIs matched)
Shape: 486 -> 136


In [9]:
glue_df.to_csv('/Users/yaochenr/project/tpd_curator/data/molecular_glues/baseline/MolGlueDB-baseline.csv', index=False)

# PROCESS PROTACS BASELINE (PROTAC-DB)

In [10]:
protac_path = '/Users/yaochenr/project/tpd_curator/data/protacs/baseline/PROTACDB-baseline-all.csv'

## Split multi-DOI rows

Some baseline rows list two DOIs in the `DOI` field (separated by `;`), e.g. when the same compound is reported in multiple papers. Expand each into one row per DOI so that downstream LLM-vs-baseline comparison can match per-paper.

In [11]:
protac_df = pd.read_csv(protac_path)
protac_df = split_multi_doi_rows(protac_df)
protac_df.head()

Split 9 multi-DOI rows; 9756 -> 9765 rows after expansion


,Record_ID,Compound_ID,Compound_Name,SMILES,InChI,InChIKey,Degradation_Target,Target_Uniprot,Recruiter,Cell_Line,Assay,DC50,DC50_units,DC50_h,Dmax,Dmax_h,Dmax_conc,DOI
0,7623,3745,BD-7148,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5CN(C6=CC=CC7=C6...,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,P25440,CRBN,MV4;11,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520
1,7624,3745,BD-7148,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5CN(C6=CC=CC7=C6...,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,P25440,CRBN,MDA-MB-231,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520
2,7625,3745,BD-7148,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5CN(C6=CC=CC7=C6...,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,P25440,CRBN,MCF-7,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520
3,7625,3745,BD-7148,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5CN(C6=CC=CC7=C6...,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,P25440,CRBN,T47D,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520
4,7647,3747,BD-9136,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5(CCN6CCN(C)CC6)...,InChI=1S/C44H44N10O5S/c1-28-47-48-37-25-59-24-...,WNTMFWJFGDGXOH-UHFFFAOYSA-N,BRD2,P25440,CRBN,MV4;11,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520


## Standardize DC50_h

In [12]:
protac_df = standardize_dc50_h(protac_df)
protac_df[['Compound_Name', 'DC50_h']].head(20)

,Compound_Name,DC50_h
0,BD-7148,4
1,BD-7148,4
2,BD-7148,4
3,BD-7148,4
4,BD-9136,4
5,BD-9136,4
6,BD-9136,4
7,BD-9136,4
8,BD-9136,4
9,BD-9136,4


## Remove rows with empty DC50 & Dmax

In [13]:
protac_df = clear_orphan_dc50_units(protac_df)
protac_df = filter_empty_dc50_dmax(protac_df)
protac_df

Cleared 125 DC50_units values where DC50 was empty
Removed 7546 rows, 2219 rows remaining


,Record_ID,Compound_ID,Compound_Name,SMILES,InChI,InChIKey,Degradation_Target,Target_Uniprot,Recruiter,Cell_Line,Assay,DC50,DC50_units,DC50_h,Dmax,Dmax_h,Dmax_conc,DOI
0,7623,3745,BD-7148,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5CN(C6=CC=CC7=C6...,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,P25440,CRBN,MV4;11,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520
1,7624,3745,BD-7148,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5CN(C6=CC=CC7=C6...,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,P25440,CRBN,MDA-MB-231,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520
2,7625,3745,BD-7148,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5CN(C6=CC=CC7=C6...,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,P25440,CRBN,MCF-7,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520
3,7625,3745,BD-7148,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5CN(C6=CC=CC7=C6...,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,P25440,CRBN,T47D,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520
4,7647,3747,BD-9136,CC1=NN=C2COCC3=C(SC(C#CC4=CN(C5(CCN6CCN(C)CC6)...,InChI=1S/C44H44N10O5S/c1-28-47-48-37-25-59-24-...,WNTMFWJFGDGXOH-UHFFFAOYSA-N,BRD2,P25440,CRBN,MV4;11,NaN,>1000,nM,4,0,4,NaN,10.1021/acs.jmedchem.3c00520
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2528,8864,5316,NaN,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,InChI=1S/C59H81N11O14S2/c1-36-50(85-34-64-36)4...,PWSQVDFJBKSXJQ-NEFBSMMZSA-N,VHL,P40337,VHL,HeLa,NaN,970,nM,12,NaN,NaN,NaN,10.1002/cbic.202200275
2529,9078,5742,NaN,CC1=C(O)C(=O)C=C2C1=CC=C1[C@@]3(C)CC[C@@]4(C)C...,"InChI=1S/C47H55N3O10/c1-26-27-11-13-33-45(4,29...",ITPPCLZSPZXHER-QQOWXAISSA-N,GRP94,P14625,CRBN,4T1,NaN,980,nM,24,NaN,NaN,NaN,10.1016/j.ejps.2023.106624
2530,3919,3722,NaN,CC1=C(C2=CC=C([C@H](C)NC(=O)[C@@H]3C[C@@H](O)C...,InChI=1S/C52H66N8O7S/c1-34(38-14-16-39(17-15-3...,FLZLBIJRCAAIIQ-MUTBYWARSA-N,NAMPT,P43490,VHL,A2780,NaN,NaN,NaN,24,NaN,NaN,NaN,10.1016/j.bmcl.2023.129393
2541,2097,5990,NaN,COC1=CC=C(/C=C\C2=CC(OC)=C(OC)C(OC)=C2)C=C1OC(...,InChI=1S/C37H39N3O10/c1-46-27-16-14-22(12-13-2...,QXCITHRZQTUQSO-SEYXRHQNSA-N,Alpha-tubulin,NaN,CRBN,A549,NaN,NaN,NaN,48,NaN,NaN,NaN,10.1016/j.ejmech.2023.116067


## Add Connectivity_Key

In [14]:
protac_df = add_connectivity_key(protac_df, 'InChIKey')
protac_df[['Compound_Name', 'InChIKey', 'Connectivity_Key']]

,Compound_Name,InChIKey,Connectivity_Key
0,BD-7148,JTXHXYXYGGEYBL-UHFFFAOYSA-N,JTXHXYXYGGEYBL
1,BD-7148,JTXHXYXYGGEYBL-UHFFFAOYSA-N,JTXHXYXYGGEYBL
2,BD-7148,JTXHXYXYGGEYBL-UHFFFAOYSA-N,JTXHXYXYGGEYBL
3,BD-7148,JTXHXYXYGGEYBL-UHFFFAOYSA-N,JTXHXYXYGGEYBL
4,BD-9136,WNTMFWJFGDGXOH-UHFFFAOYSA-N,WNTMFWJFGDGXOH
...,...,...,...
2528,NaN,PWSQVDFJBKSXJQ-NEFBSMMZSA-N,PWSQVDFJBKSXJQ
2529,NaN,ITPPCLZSPZXHER-QQOWXAISSA-N,ITPPCLZSPZXHER
2530,NaN,FLZLBIJRCAAIIQ-MUTBYWARSA-N,FLZLBIJRCAAIIQ
2541,NaN,QXCITHRZQTUQSO-SEYXRHQNSA-N,QXCITHRZQTUQSO


## Save data

In [15]:
protac_df.to_csv('/Users/yaochenr/project/tpd_curator/data/protacs/baseline/PROTACDB-baseline-cleaned.csv', index=False)

## Remove DOIs not full-text avaliable in PMC

In [16]:
excluded_txt_path = "/Users/yaochenr/project/molecular_glue_extractor/data_source/250825_protac_papers/excluded_dois.txt"
excluded_csv_path = "/Users/yaochenr/project/molecular_glue_extractor/data_source/250825_protac_papers/excluded_dois.csv"

excluded_dois_df = convert_excluded_dois_txt_to_csv(excluded_txt_path, excluded_csv_path)
excluded_dois_df.head()

Saved excluded DOIs to: /Users/yaochenr/project/molecular_glue_extractor/data_source/250825_protac_papers/excluded_dois.csv
Shape: (423, 2)
exclude_reason
NOT_FOUND_IN_PMC_DATABASE     261
IN_PMC_BUT_NOT_OPEN_ACCESS    162
Name: count, dtype: int64


,DOI,exclude_reason
0,10.1021/acs.jmedchem.8b01572,NOT_FOUND_IN_PMC_DATABASE
1,10.1039/c8cc09541h,NOT_FOUND_IN_PMC_DATABASE
2,10.1016/j.chembiol.2023.01.007,NOT_FOUND_IN_PMC_DATABASE
3,10.1039/c9cc08238g,NOT_FOUND_IN_PMC_DATABASE
4,10.1021/acs.jmedchem.1c01774,NOT_FOUND_IN_PMC_DATABASE


In [17]:
protac_df = remove_excluded_dois(protac_df, excluded_csv_path)

Removed 1847 rows (262 unique excluded DOIs matched)
Shape: 2219 -> 372


In [19]:
protac_df.to_csv('/Users/yaochenr/project/tpd_curator/data/protacs/baseline/PROTACDB-baseline.csv', index=False)